In [13]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/tiwarikanaksuraj/train-csv/train (1).csv


In [14]:
df=pd.read_csv("/kaggle/input/datasets/tiwarikanaksuraj/train-csv/train (1).csv")


# milestone-1 
#### Q1 Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?  

In [15]:
x=df['answer'].value_counts().max() +  df['answer'].value_counts().min()

print("Sum of Most frequent and least frequent options in answer column =",x)

Sum of Most frequent and least frequent options in answer column = 814


### Q2 After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?   

In [16]:
import string
has_punctuation = df["prompt"].apply(
    lambda text: any(c in string.punctuation for c in text)
)
print(has_punctuation.unique())

[ True]


In [17]:
import string
def clean_prompt(text):
    text=text.lower()
    translator=str.maketrans('','',string.punctuation) #nothing,nothing,delete punctuation 
    text=text.translate(translator)
    return text
    

In [18]:
df["prompt"]=df["prompt"].apply(clean_prompt)
#verifying 
has_punctuation = df["prompt"].apply(
    lambda text: any(c in string.punctuation for c in text)
)
print(has_punctuation.unique())

[False]


In [19]:
vocab=set()
for text in df["prompt"]:
    vocab.update(text.split())

print(len(vocab))

859


### Q3 Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?  


In [20]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import string

temp=df[df["id"]==1]["prompt"].iloc[0].split()
filtered_words=[i for i in temp if i not in ENGLISH_STOP_WORDS] 
print("unfiltered string :",temp)
print("filtered string list : ",filtered_words)
print("length of list ",len(filtered_words))

unfiltered string : ['pick', 'the', 'best', 'possible', 'answer', 'what', 'is', 'martin', 'heideggers', 'view', 'on', 'the', 'relationship', 'between', 'time', 'and', 'human', 'existence', 'among', 'the', 'listed', 'options']
filtered string list :  ['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']
length of list  13


### Q4 Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
combinelist=pd.concat(
    [df["prompt"],
    df["A"],
    df["B"],
     df["C"],
     df["D"],
     df["E"]
    ])
vectorizer=TfidfVectorizer(stop_words="english")
vectorizer.fit(combinelist)

print(len(vectorizer.vocabulary_))


2807


### Q5 Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).  

In [22]:
from sklearn.metrics.pairwise import cosine_similarity
row=df[df["id"]==1].iloc[0]
prompt=row["prompt"]
optA=row["A"]
prompt_vec=vectorizer.transform([prompt])
option_vec=vectorizer.transform([optA])
print(f"Similarity score : {cosine_similarity(prompt_vec,option_vec)[0,0]:.4f}")

Similarity score : 0.1682


### Q6 For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.  

In [23]:
def transform_columns(df, vectorizer):
    return (
        vectorizer.transform(df['prompt']),
        vectorizer.transform(df['A']),
        vectorizer.transform(df['B']),
        vectorizer.transform(df['C']),
        vectorizer.transform(df['D']),
        vectorizer.transform(df['E'])
    )

    
prompt_vec, A_vec, B_vec, C_vec, D_vec, E_vec = transform_columns(df, vectorizer)

prompt_vec.shape #where rows,no_of_unnique_words

(2000, 2807)

In [24]:
from sklearn.metrics.pairwise import cosine_similarity

scores = {
    'A': cosine_similarity(prompt_vec, A_vec).diagonal(),
    'B': cosine_similarity(prompt_vec, B_vec).diagonal(),
    'C': cosine_similarity(prompt_vec, C_vec).diagonal(),
    'D': cosine_similarity(prompt_vec, D_vec).diagonal(),
    'E': cosine_similarity(prompt_vec, E_vec).diagonal()
}

correct_pred=0
for i in range(len(scores["A"])):
    row_scores={
        'A': scores['A'][i],
        'B': scores['B'][i],
        'C': scores['C'][i],
        'D': scores['D'][i],
        'E': scores['E'][i]  
    }
    best_opt='A'
    for option in ['B','C','D','E']:
        if row_scores[option] > row_scores[best_opt]:
            best_opt=option

    if (best_opt == df.loc[i,'answer']):
        correct_pred+=1

accuracy=(correct_pred/df.shape[0])*100
print(f"Accuracy of cosimilarity : {accuracy}")

Accuracy of cosimilarity : 12.5


### Q9 The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [25]:
def map3(y_true, top3_preds):
    score = 0

    for actual, pred in zip(y_true, top3_preds):
        if actual == pred[0]:
            score += 1.0

        elif actual == pred[1]:
            score += 0.5

        elif actual == pred[2]:
            score += 1/3

    return score / len(y_true)

In [26]:
counts = df['answer'].value_counts()

top3 = counts.index[:3].tolist()

print(top3)

['B', 'C', 'A']


In [27]:
y_pred = [top3] * len(df)

In [28]:
print("Majority class baseline : ",round(map3(df["answer"],y_pred),3))

Majority class baseline :  0.421


In [29]:
y_pred=[]
for i in range(len(scores["A"])):
    row_scores={
        'A': scores['A'][i],
        'B': scores['B'][i],
        'C': scores['C'][i],
        'D': scores['D'][i],
        'E': scores['E'][i]  
    }
    top3=[]
   
    for k in range(3):
        best_opt=None
        best_score=-1
        for option,score in row_scores.items():
            if score > best_score:
                best_score=score
                best_opt=option
        top3.append(best_opt)
        row_scores.pop(best_opt)
    
    y_pred.append(top3)    
print(len(y_pred))

2000


In [31]:
print(round(map3(df['answer'], y_pred),2))

0.29
